In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
import os
os.chdir("/content")
print(os.getcwd())


/content


In [3]:
!rm -rf /content/Task-driven-low-light-enhancement
!git clone https://github.com/SarthakBaghel/Task-driven-low-light-enhancement.git /content/Task-driven-low-light-enhancement
%cd /content/Task-driven-low-light-enhancement
!pip install -r requirements.txt


Cloning into '/content/Task-driven-low-light-enhancement'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 68 (delta 12), reused 63 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 150.09 KiB | 1.21 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/Task-driven-low-light-enhancement
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 16.2 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


In [5]:
from pathlib import Path
import os
import shutil

DRIVE_ROOT = Path("/content/drive/MyDrive")
PIPELINE_ROOT = DRIVE_ROOT / "task_driven_video_pipeline"
KAGGLE_V1_ROOT = PIPELINE_ROOT / "kaggle_v1"
CHECKPOINTS_ROOT = DRIVE_ROOT / "task_driven_checkpoints" / "kaggle_v1"

RUN_TAG = "subject_class_balanced_20k"

TRAIN_CLEAN_ROOT = KAGGLE_V1_ROOT / f"train_clean_{RUN_TAG}"
VAL_CLEAN_ROOT = KAGGLE_V1_ROOT / f"val_clean_{RUN_TAG}"
TEST_CLEAN_ROOT = KAGGLE_V1_ROOT / f"test_clean_{RUN_TAG}"

FULL_BUNDLE_ROOT = Path(f"/content/kaggle_v1_full_clean_bundle_{RUN_TAG}")
FULL_TRAIN_LINK = FULL_BUNDLE_ROOT / "train"
FULL_VAL_LINK = FULL_BUNDLE_ROOT / "val"

FULL_CKPT_DIR = CHECKPOINTS_ROOT / f"full_clean_detector_{RUN_TAG}_resnet18"
FULL_BEST_CKPT = FULL_CKPT_DIR / f"kaggle_v1_clean_full_{RUN_TAG}_best.pt"
FULL_LAST_CKPT = FULL_CKPT_DIR / f"kaggle_v1_clean_full_{RUN_TAG}_last.pt"

VAL_REPORT_DIR = KAGGLE_V1_ROOT / f"eval_full_clean_on_val_{RUN_TAG}"
TEST_REPORT_DIR = KAGGLE_V1_ROOT / f"eval_full_clean_on_test_{RUN_TAG}"

assert TRAIN_CLEAN_ROOT.exists(), f"Missing: {TRAIN_CLEAN_ROOT}"
assert VAL_CLEAN_ROOT.exists(), f"Missing: {VAL_CLEAN_ROOT}"
assert TEST_CLEAN_ROOT.exists(), f"Missing: {TEST_CLEAN_ROOT}"

print("TRAIN_CLEAN_ROOT:", TRAIN_CLEAN_ROOT)
print("VAL_CLEAN_ROOT:", VAL_CLEAN_ROOT)
print("TEST_CLEAN_ROOT:", TEST_CLEAN_ROOT)
print("FULL_BUNDLE_ROOT:", FULL_BUNDLE_ROOT)
print("FULL_CKPT_DIR:", FULL_CKPT_DIR)


TRAIN_CLEAN_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/train_clean_subject_class_balanced_20k
VAL_CLEAN_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/val_clean_subject_class_balanced_20k
TEST_CLEAN_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/test_clean_subject_class_balanced_20k
FULL_BUNDLE_ROOT: /content/kaggle_v1_full_clean_bundle_subject_class_balanced_20k
FULL_CKPT_DIR: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/full_clean_detector_subject_class_balanced_20k_resnet18


In [6]:
!find "{TRAIN_CLEAN_ROOT}/open" -type f | wc -l
!find "{TRAIN_CLEAN_ROOT}/closed" -type f | wc -l

!find "{VAL_CLEAN_ROOT}/open" -type f | wc -l
!find "{VAL_CLEAN_ROOT}/closed" -type f | wc -l

!find "{TEST_CLEAN_ROOT}/open" -type f | wc -l
!find "{TEST_CLEAN_ROOT}/closed" -type f | wc -l


383
383
973
973
8644
8644


In [7]:
if FULL_BUNDLE_ROOT.exists():
    shutil.rmtree(FULL_BUNDLE_ROOT)

FULL_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)

os.symlink(TRAIN_CLEAN_ROOT, FULL_TRAIN_LINK)
os.symlink(VAL_CLEAN_ROOT, FULL_VAL_LINK)

print("Created bundle root:", FULL_BUNDLE_ROOT)
print("train ->", FULL_TRAIN_LINK.resolve())
print("val   ->", FULL_VAL_LINK.resolve())


Created bundle root: /content/kaggle_v1_full_clean_bundle_subject_class_balanced_20k
train -> /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/train_clean_subject_class_balanced_20k
val   -> /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/val_clean_subject_class_balanced_20k


In [8]:
!find "{FULL_BUNDLE_ROOT}" -maxdepth 2 -type d | sed -n '1,40p'


/content/kaggle_v1_full_clean_bundle_subject_class_balanced_20k


In [9]:
!python3 /content/Task-driven-low-light-enhancement/train_transfer_detector.py \
  {FULL_BUNDLE_ROOT} \
  --runtime colab \
  --colab-workspace-root /content/Task-driven-low-light-enhancement \
  --backbone resnet18 \
  --epochs 15 \
  --batch-size 64 \
  --num-workers 2 \
  --learning-rate 1e-4 \
  --monitor f1 \
  --threshold-objective f1 \
  --threshold-candidates 0.2 0.3 0.4 0.5 0.6 0.7 \
  --early-stopping-patience 5 \
  --scheduler-patience 2 \
  --deterministic \
  --save-path artifacts/kaggle_v1_full/kaggle_v1_clean_full_subject_class_balanced_20k_best.pt \
  --save-last-path artifacts/kaggle_v1_full/kaggle_v1_clean_full_subject_class_balanced_20k_last.pt \
  --drive-checkpoint-dir {FULL_CKPT_DIR}


Adjusted initial batch size from 64 to 24 based on available GPU memory.
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /content/Task-driven-low-light-enhancement/artifacts/torch_cache/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 191MB/s]
Runtime: colab
Workspace root: /content/Task-driven-low-light-enhancement
Training on device: cuda
Classes: {'open': 0, 'closed': 1}
Samples: train=766 | val=1946
Focal alpha: [1.0, 1.0]
Best checkpoint path: /content/Task-driven-low-light-enhancement/artifacts/kaggle_v1_full/kaggle_v1_clean_full_subject_class_balanced_20k_best.pt
Last checkpoint path: /content/Task-driven-low-light-enhancement/artifacts/kaggle_v1_full/kaggle_v1_clean_full_subject_class_balanced_20k_last.pt
Drive backup directory: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/full_clean_detector_subject_class_balanced_20k_resnet18
                                                          
Epoch 1/15 | lr=0.000100
Train    los

In [10]:
!ls {FULL_BEST_CKPT}
!ls {FULL_LAST_CKPT}


/content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/full_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_full_subject_class_balanced_20k_best.pt
/content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/full_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_full_subject_class_balanced_20k_last.pt


In [11]:
!python3 /content/Task-driven-low-light-enhancement/evaluate_transfer_detector.py \
  {FULL_BEST_CKPT} \
  {VAL_CLEAN_ROOT} \
  --batch-size 64 \
  --num-workers 0 \
  --output-dir {VAL_REPORT_DIR}


Evaluating on device: cuda
Classes: {'open': 0, 'closed': 1}
Clean    loss: 0.0465 | accuracy: 0.9697 | precision: 0.9534 | recall: 0.9877 | f1: 0.9702 | closed_recall: 0.9877 | threshold: 0.70
Saved evaluation report to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_full_clean_on_val_subject_class_balanced_20k

Report Table
| Dataset | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Clean | 96.97% | 95.34% | 98.77% | 97.02% |

Saved compact CSV to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_full_clean_on_val_subject_class_balanced_20k/experiment_results.csv
Saved detailed CSV to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_full_clean_on_val_subject_class_balanced_20k/evaluation_results.csv


In [12]:
!python3 /content/Task-driven-low-light-enhancement/evaluate_transfer_detector.py \
  {FULL_BEST_CKPT} \
  {TEST_CLEAN_ROOT} \
  --batch-size 64 \
  --num-workers 0 \
  --output-dir {TEST_REPORT_DIR}


Evaluating on device: cuda
Classes: {'open': 0, 'closed': 1}
Clean    loss: 0.1234 | accuracy: 0.9259 | precision: 0.8762 | recall: 0.9919 | f1: 0.9305 | closed_recall: 0.9919 | threshold: 0.70
Saved evaluation report to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_full_clean_on_test_subject_class_balanced_20k

Report Table
| Dataset | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Clean | 92.59% | 87.62% | 99.19% | 93.05% |

Saved compact CSV to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_full_clean_on_test_subject_class_balanced_20k/experiment_results.csv
Saved detailed CSV to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_full_clean_on_test_subject_class_balanced_20k/evaluation_results.csv


In [13]:
!sed -n '1,200p' {VAL_REPORT_DIR}/evaluation_summary.txt
!sed -n '1,200p' {TEST_REPORT_DIR}/evaluation_summary.txt


Using threshold=0.70 for the closed-eye class.
Closed-eye recall on clean data: 0.9877.Using threshold=0.70 for the closed-eye class.
Closed-eye recall on clean data: 0.9919.

In [14]:
from pathlib import Path
import pandas as pd

paths = {
    "val": Path("/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_full_clean_on_val_subject_class_balanced_20k/experiment_results.csv"),
    "test": Path("/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_full_clean_on_test_subject_class_balanced_20k/experiment_results.csv"),
}

for name, path in paths.items():
    print(f"\n=== {name.upper()} ===")
    print("Path:", path)

    if not path.exists():
        print("Missing file.")
        continue

    df = pd.read_csv(path)
    print(df.to_string(index=False))



=== VAL ===
Path: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_full_clean_on_val_subject_class_balanced_20k/experiment_results.csv
Dataset  Accuracy  Precision  Recall    F1
  Clean     96.97      95.34   98.77 97.02

=== TEST ===
Path: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_full_clean_on_test_subject_class_balanced_20k/experiment_results.csv
Dataset  Accuracy  Precision  Recall    F1
  Clean     92.59      87.62   99.19 93.05


In [15]:
print("Phase 4 complete.")
print("FULL_BEST_CKPT:", FULL_BEST_CKPT)
print("FULL_LAST_CKPT:", FULL_LAST_CKPT)
print("VAL_REPORT_DIR:", VAL_REPORT_DIR)
print("TEST_REPORT_DIR:", TEST_REPORT_DIR)


Phase 4 complete.
FULL_BEST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/full_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_full_subject_class_balanced_20k_best.pt
FULL_LAST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/full_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_full_subject_class_balanced_20k_last.pt
VAL_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_full_clean_on_val_subject_class_balanced_20k
TEST_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_full_clean_on_test_subject_class_balanced_20k
